In [ ]:
import os, csv, requests

CATEGORY = "Category:Borobudur"
OUT      = os.path.expanduser("/mnt/c/Users/Alpha/Pictures/vggt")
WIDTH    = 1600                      # cukup besar; VGGT toh resize ke 518
HEADERS  = {"User-Agent": "vggt-research/0.1 (alphasuthamedia@gmail.com)"}  # wajib menurut aturan API Wikimedia
API      = "https://commons.wikimedia.org/w/api.php"

os.makedirs(OUT, exist_ok=True)
params = {"action": "query", "format": "json", "generator": "categorymembers",
          "gcmtitle": CATEGORY, "gcmtype": "file", "gcmlimit": 50,
          "prop": "imageinfo", "iiprop": "url|extmetadata", "iiurlwidth": WIDTH}

rows = []
while True:
    r = requests.get(API, params=params, headers=HEADERS).json()
    for p in r.get("query", {}).get("pages", {}).values():
        info = p["imageinfo"][0]
        if not p["title"].lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        meta = info.get("extmetadata", {})
        name = p["title"].replace("File:", "").replace("/", "_")
        img  = requests.get(info.get("thumburl", info["url"]), headers=HEADERS).content
        open(os.path.join(OUT, name), "wb").write(img)
        rows.append([name, meta.get("Artist", {}).get("value", ""),
                     meta.get("LicenseShortName", {}).get("value", ""), info["descriptionurl"]])
    if "continue" not in r:
        break
    params.update(r["continue"])

with open(os.path.join(OUT, "_atribusi.csv"), "w", newline="") as f:
    csv.writer(f).writerows([["file", "artist", "license", "url"]] + rows)
print(len(rows), "foto diunduh ke", OUT)